# COVER-KBC Profile F1 TEST Submission Builder

This notebook builds the promoted Profile F1 submission path that was actually leaderboard-tested:

1. Start from the already-winning Profile E3 475-row prediction artifact.
2. Run `MistralStockEmptyRescue` only for empty `companyTradesAtStockExchange` rows.
3. Merge the 100 stock rows back into the 475-row baseline with fail-closed checks.
4. Package exactly one `predictions.jsonl` member for submission.

Do not use a fresh `run_cover.py` full TEST inference run to reproduce the 0.5878 score. A full rerun recomputes all relations and can produce a different valid submission with a lower score.

In [ ]:
# CELL 0 - Runtime paths
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/vquclinh/FactElicit-AKBC"
REPO = Path("/content/FactElicit-AKBC")
CONFIG_PATH = REPO / "configs/experiments/cover_kbc_v3_8_profile_f1_stock_empty_rescue_test.yaml"
TEST_PATH = REPO / "benchmark/data/test.jsonl"

# Upload or copy the hidden-winning Profile E3 475-row artifact to this path.
BASELINE_E3_PATH = Path("/content/profile_e3_best_predictions.jsonl")

TARGET_DIR = Path("/content/f1_stock_empty_rescue")
MERGED_PATH = Path("/content/profile_f1_stock_empty_rescue_475.jsonl")
SUBMISSION_ZIP = Path("/content/profile_f1_stock_empty_rescue_submission.zip")
DRIVE_OUT = Path("/content/drive/MyDrive/cover_kbc/profile_f1_stock_empty_rescue")

print("Repo:", REPO)
print("Config:", CONFIG_PATH)
print("Baseline E3 artifact:", BASELINE_E3_PATH)
print("Targeted repair output:", TARGET_DIR)
print("Merged predictions:", MERGED_PATH)
print("Submission zip:", SUBMISSION_ZIP)
print("Drive output directory:", DRIVE_OUT)

In [ ]:
# CELL 0B - Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("Drive output directory:", DRIVE_OUT)

In [ ]:
# CELL 1 - Clone or pull the latest repo code
import os
import subprocess
import sys

if REPO.exists():
    os.chdir(REPO)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    os.chdir(REPO)

print("cwd:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["git", "status", "--short"], check=True)

In [ ]:
# CELL 2 - Install package and model dependencies
import os
import subprocess
import sys
from pathlib import Path

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hf]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "mistral-common>=1.6.2"], check=True)

src_path = str(REPO / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import cover_kbc
print("cover_kbc:", cover_kbc.__file__)

In [ ]:
# CELL 3 - Hugging Face login
import os
import getpass
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as exc:
    print("Colab secret lookup failed; falling back to env/manual token:", repr(exc))
    token = None

token = token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token:
    token = getpass.getpass("Enter HF_TOKEN: ")

assert token and token.strip(), "HF_TOKEN is required to load the gated model."
os.environ["HF_TOKEN"] = token.strip()
os.environ["HUGGING_FACE_HUB_TOKEN"] = token.strip()
login(token=token.strip(), add_to_git_credential=False)
print("HF login completed.")

In [ ]:
# CELL 4 - Static preflight: config, model portfolio, and baseline artifact shape
import hashlib
import json
from collections import Counter
from pathlib import Path
import yaml

os.chdir(REPO)
assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"
assert TEST_PATH.exists(), f"Missing official TEST split: {TEST_PATH}"
assert BASELINE_E3_PATH.exists(), (
    "Upload/copy the winning Profile E3 475-row file to "
    f"{BASELINE_E3_PATH} before continuing."
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

cfg = yaml.safe_load(CONFIG_PATH.read_text())
assert cfg["experiment"]["name"] == "cover_kbc_v3_8_profile_f1_stock_empty_rescue_test"
assert cfg["experiment"]["frozen_baseline"]["hidden_test_overall_f1"] == 0.5878
assert cfg["leaderboard_repair"]["features"]["mistral_stock_empty_rescue"] is True
assert "Qwen" not in json.dumps(cfg.get("model_profile", {}), sort_keys=True)

baseline = read_jsonl(BASELINE_E3_PATH)
test_rows = read_jsonl(TEST_PATH)
assert len(baseline) == 475, f"Baseline must have 475 rows, found {len(baseline)}"
assert len(test_rows) == 475, f"TEST split must have 475 rows, found {len(test_rows)}"

baseline_keys = [(r["SubjectEntity"], r["Relation"]) for r in baseline]
test_keys = [(r["SubjectEntity"], r["Relation"]) for r in test_rows]
assert baseline_keys == test_keys, "Baseline row identities/order do not match official TEST."

relation_counts = Counter(r["Relation"] for r in baseline)
stock_rows = [r for r in baseline if r["Relation"] == "companyTradesAtStockExchange"]
empty_stock = sum(not r.get("ObjectEntities") for r in stock_rows)

print("Config OK:", CONFIG_PATH)
print("Baseline rows:", len(baseline))
print("Baseline sha256:", sha256_file(BASELINE_E3_PATH))
print("Relation counts:", dict(sorted(relation_counts.items())))
print("Stock rows:", len(stock_rows))
print("Empty stock rows to attempt:", empty_stock)
if empty_stock != 57:
    print("WARNING: expected the historical E3 artifact to have 57 empty stock rows; verify this is the exact E3 best artifact.")

In [ ]:
# CELL 5 - Dry-run the targeted stock rescue invariants without loading the model
import shutil
import subprocess
import sys

os.chdir(REPO)
DRY_DIR = Path("/content/f1_stock_empty_rescue_dry_run")
if DRY_DIR.exists():
    shutil.rmtree(DRY_DIR)

cmd = [
    sys.executable,
    "scripts/run_stock_empty_rescue.py",
    "--config",
    str(CONFIG_PATH),
    "--baseline-predictions",
    str(BASELINE_E3_PATH),
    "--split",
    "test",
    "--output-dir",
    str(DRY_DIR),
    "--dry-run",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print((DRY_DIR / "stock_empty_rescue_dry_run.json").read_text())

In [ ]:
# CELL 6 - Run targeted Stock Empty Rescue only
# This does NOT run the full 475-row pipeline. It loads Mistral once and spends
# exactly 4 calls for each empty stock row in the uploaded E3 baseline artifact.
import shutil
import subprocess
import sys

os.chdir(REPO)
if TARGET_DIR.exists():
    raise FileExistsError(
        f"{TARGET_DIR} already exists. Move/delete it manually if you intentionally want a fresh targeted run."
    )

cmd = [
    sys.executable,
    "-u",
    "scripts/run_stock_empty_rescue.py",
    "--config",
    str(CONFIG_PATH),
    "--baseline-predictions",
    str(BASELINE_E3_PATH),
    "--split",
    "test",
    "--output-dir",
    str(TARGET_DIR),
]
print("Running:", " ".join(cmd), flush=True)
completed = subprocess.run(cmd, text=True)
print("Return code:", completed.returncode)
if completed.returncode != 0:
    raise SystemExit(completed.returncode)

In [ ]:
# CELL 7 - Merge the repaired 100 stock rows into the 475-row E3 baseline
import subprocess
import sys

os.chdir(REPO)
TARGETED_RESULTS = TARGET_DIR / "stock_empty_rescue_results.jsonl"
assert TARGETED_RESULTS.exists(), f"Missing targeted stock results: {TARGETED_RESULTS}"
if MERGED_PATH.exists():
    raise FileExistsError(f"{MERGED_PATH} already exists. Move/delete it before rebuilding.")

cmd = [
    sys.executable,
    "scripts/merge_targeted_relation_results.py",
    "--baseline-predictions",
    str(BASELINE_E3_PATH),
    "--targeted-results",
    str(TARGETED_RESULTS),
    "--relation",
    "companyTradesAtStockExchange",
    "--expected-targeted-rows",
    "100",
    "--output",
    str(MERGED_PATH),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Merged predictions:", MERGED_PATH)

In [ ]:
# CELL 8 - Validate the merged 475-row file and inspect repair accounting
import json
from collections import Counter

baseline = read_jsonl(BASELINE_E3_PATH)
merged = read_jsonl(MERGED_PATH)
assert len(merged) == 475
assert [(r["SubjectEntity"], r["Relation"]) for r in merged] == [(r["SubjectEntity"], r["Relation"]) for r in baseline]

changed = []
for before, after in zip(baseline, merged):
    if before != after:
        changed.append((before["SubjectEntity"], before["Relation"], before.get("ObjectEntities") or [], after.get("ObjectEntities") or []))

changed_relations = sorted({rel for _, rel, _, _ in changed})
assert set(changed_relations).issubset({"companyTradesAtStockExchange"}), changed_relations

relation_counts = Counter(r["Relation"] for r in merged)
empty_by_relation = {
    rel: sum(not r.get("ObjectEntities") for r in merged if r["Relation"] == rel)
    for rel in sorted(relation_counts)
}

print("Merged rows:", len(merged))
print("Changed rows:", len(changed))
print("Changed relations:", changed_relations)
print("Relation counts:", dict(sorted(relation_counts.items())))
print("Empty rows by relation:", empty_by_relation)
print("Merged sha256:", sha256_file(MERGED_PATH))

accounting_path = TARGET_DIR / "stock_empty_rescue_accounting.json"
assert accounting_path.exists(), f"Missing accounting: {accounting_path}"
accounting = json.loads(accounting_path.read_text())
print(json.dumps({
    "empty_stock_rows_attempted": accounting["empty_stock_rows_attempted"],
    "rescued_empty_rows": accounting["rescued_empty_rows"],
    "kept_empty_rows": accounting["kept_empty_rows"],
    "total_stock_empty_rescue_calls": accounting["total_stock_empty_rescue_calls"],
    "model_id": accounting["model_id"],
    "model_revision": accounting["model_revision"],
}, indent=2))

In [ ]:
# CELL 9 - Package the TEST submission zip
import subprocess
import sys

os.chdir(REPO)
if SUBMISSION_ZIP.exists():
    raise FileExistsError(f"{SUBMISSION_ZIP} already exists. Move/delete it before packaging again.")

cmd = [
    sys.executable,
    "scripts/package_submission.py",
    "--input",
    str(MERGED_PATH),
    "--split",
    "test",
    "--output",
    str(SUBMISSION_ZIP),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Submit this zip:", SUBMISSION_ZIP)
print("Zip sha256:", sha256_file(SUBMISSION_ZIP))

In [ ]:
# CELL 10 - Save artifacts to Google Drive
import json
import shutil

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR = DRIVE_OUT / TARGET_DIR.name
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(TARGET_DIR, DRIVE_RUN_DIR)

shutil.copy2(BASELINE_E3_PATH, DRIVE_OUT / BASELINE_E3_PATH.name)
shutil.copy2(MERGED_PATH, DRIVE_OUT / MERGED_PATH.name)
shutil.copy2(SUBMISSION_ZIP, DRIVE_OUT / SUBMISSION_ZIP.name)
provenance_path = MERGED_PATH.with_name(MERGED_PATH.stem + "_provenance.json")
if provenance_path.exists():
    shutil.copy2(provenance_path, DRIVE_OUT / provenance_path.name)

summary = {
    "baseline_e3_path": str(BASELINE_E3_PATH),
    "baseline_e3_sha256": sha256_file(BASELINE_E3_PATH),
    "target_dir": str(TARGET_DIR),
    "merged_predictions": str(MERGED_PATH),
    "merged_sha256": sha256_file(MERGED_PATH),
    "submission_zip": str(SUBMISSION_ZIP),
    "submission_zip_sha256": sha256_file(SUBMISSION_ZIP),
    "drive_dir": str(DRIVE_OUT),
}
(DRIVE_OUT / "profile_f1_stock_empty_rescue_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("Saved run directory to:", DRIVE_RUN_DIR)
print("Saved merged predictions to:", DRIVE_OUT / MERGED_PATH.name)
print("Saved submission zip to:", DRIVE_OUT / SUBMISSION_ZIP.name)
print("Saved summary to:", DRIVE_OUT / "profile_f1_stock_empty_rescue_summary.json")

In [ ]:
# CELL 11 - Download artifacts from the browser session
from google.colab import files

files.download(str(SUBMISSION_ZIP))
files.download(str(MERGED_PATH))
files.download(str(TARGET_DIR / "stock_empty_rescue_accounting.json"))

In [ ]:
# CELL 12 - Disconnect runtime after artifacts are saved
from google.colab import runtime

runtime.unassign()